# `StructurePreprocessor.encode_pae()`

`encode_pae` reads an AlphaFold PAE sidecar JSON and collapses the L×L predicted-aligned-error matrix into per-residue summaries (`pae_row_mean/min/max`, `pae_local_mean` / `pae_distal_mean` within / beyond `±local_window`, `pae_asymmetry`, `pae_band_means`), all normalized to `[0, 1]` by AlphaFold's 31.75 Å saturation cap.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import aaanalysis as aa
import aaanalysis.utils as ut
aa.options['verbose'] = False
warnings.filterwarnings('ignore')

PDB_FIXTURES = Path(aa.__file__).resolve().parent / '_data' / 'pdb_test'
strp = aa.StructurePreprocessor(verbose=False)
df_seq = pd.DataFrame({'entry': ['AF_TINY'],
                       'sequence': ['ACDEFGHIKLMNPQRSTVWYACDEFGHIKL']})
import tempfile, shutil
pae_dir = tempfile.mkdtemp()
shutil.copy(PDB_FIXTURES / 'AF_TINY_pae.json',
            Path(pae_dir) / 'AF_TINY.json')

feats = ['pae_row_mean', 'pae_local_mean', 'pae_distal_mean',
         'pae_asymmetry']
dict_pae = strp.encode_pae(df_seq=df_seq, pae_folder=pae_dir,
                          features=feats, local_window=5)
arr = dict_pae['AF_TINY']
print('shape (L, D):', arr.shape)
print('mean local vs distal PAE:',
      round(float(np.nanmean(arr[:, 1])), 3), 'vs',
      round(float(np.nanmean(arr[:, 2])), 3))

shape (L, D): (30, 4)
mean local vs distal PAE: 0.087 vs 0.361


**Further parameters.** ``StructurePreprocessor.encode_pae`` also accepts: ``pae_band_edges`` — Used by ``pae_band_means`` only; ``on_failure`` — What to do for entries whose PAE load fails (missing sidecar, malformed JSON, shape mismatch); ``return_df`` — If ``True``, also return the per-row status DataFrame as a second element ``(dict_num, df_seq_out)``.

In [2]:
# Further parameters: ``pae_band_edges`` sets the distance bands used by the
# ``pae_band_means`` feature, ``on_failure`` governs entries whose PAE load
# fails, and ``return_df=True`` also returns a per-row status frame (``pae_ok``).
dict_pae_band, df_pae_status = strp.encode_pae(
    df_seq=df_seq, pae_folder=pae_dir,
    features=['pae_row_mean', 'pae_band_means'],
    pae_band_edges=(5, 15), on_failure='nan', return_df=True)
aa.display_df(df_pae_status, n_rows=10, show_shape=True)

DataFrame shape: (1, 3)


,entry,sequence,pae_ok
1,AF_TINY,ACDEFGHIKLMNPQRSTVWYACDEFGHIKL,True
